In [2]:
import pandas as pd

df = pd.read_parquet("../data/interim/pool_raw.parquet")   # <- your file
TS, SRC, KEY, LABEL = "timestamp", "source", "incident_key", "is_malicious"  # <- your column names
df[TS] = pd.to_datetime(df[TS], utc=True)

cuts = {
    "cisco_asa": {
        "train": ("2026-07-26 11:00", "2026-07-26 14:00"),
        "val":   ("2026-07-26 17:00", "2026-07-26 21:00"),
        "test":  ("2026-07-27 00:00", "2026-07-27 07:00"),
    },
    "aws_vpc_flow_log": {
        "train": ("2026-07-26 00:00", "2026-07-28 00:00"),
        "val":   ("2026-07-29 16:00", "2026-07-29 19:00"),
        "test":  ("2026-07-29 20:00", "2026-07-31 00:00"),
    },
}

In [3]:
df["split"] = "dropped"
for src, sp in cuts.items():
    for name, (a, b) in sp.items():
        m = (df[SRC] == src) & (df[TS] >= pd.Timestamp(a, tz="UTC")) & (df[TS] < pd.Timestamp(b, tz="UTC"))
        df.loc[m, "split"] = name

KeyError: 'source'

In [24]:
print(df.columns.tolist())
df.head()

['timestamp', 'org_id', 'label_binary', 'message_type', 'stream_name', 'product_name', 'action', 'protocol', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'src_host', 'dst_host', 'username', 'severity', 'vendor_code', 'message_sanitized', 'day', 'incident_ids', 'split']


,timestamp,org_id,label_binary,message_type,stream_name,product_name,action,protocol,src_ip,dst_ip,...,dst_port,src_host,dst_host,username,severity,vendor_code,message_sanitized,day,incident_ids,split
0,2024-07-26 11:14:11+00:00,ORG-0001,suspicious,auth_failure,generic_log,Precinct,,,,,...,,10.33.86.71,,,informational,,<142>Jul 26 11:14:02 local1: Host: HOST-0006 D...,07-26,[],dropped
1,2024-07-26 11:36:12+00:00,ORG-0001,suspicious,auth_failure,generic_log,Precinct,,,,,...,,10.33.86.71,,,informational,,<142>Jul 26 11:36:04 local1: Host: HOST-0006 D...,07-26,[],dropped
2,2024-07-26 11:45:27+00:00,ORG-0001,suspicious,auth_failure,generic_log,Precinct,,,,,...,,10.33.86.71,,,informational,,<142>Jul 26 11:45:21 local1: Host: HOST-0002 D...,07-26,[],dropped
3,2024-07-27 03:27:32+00:00,ORG-0003,suspicious,auth_failure,microsoft-windows-security-auditing,Windows Active Directory,Logon,,192.168.231.254,,...,,host-0015.example.internal,,MACHINE-0021$,Info,6069773856,"WitFoo-WinLogBeat ::: {""@metadata"":{""beat"":""wi...",07-27,[],dropped
4,2024-07-27 03:27:13+00:00,ORG-0003,suspicious,auth_failure,microsoft-windows-security-auditing,Windows Active Directory,Logon,,192.168.233.228,,...,,host-0016.example.internal,,MACHINE-0458$,Info,1356448752,"WitFoo-WinLogBeat ::: {""@metadata"":{""beat"":""wi...",07-27,[],dropped


In [23]:
SRC = "stream_name"
TS = "timestamp"

In [22]:
   df["timestamp"] = pd.to_datetime(
       df["timestamp"].dt.tz_localize(None).astype("int64"), unit="s", utc=True
   )
   print(df["timestamp"].min(), df["timestamp"].max())

2024-07-26 11:10:48+00:00 2024-08-01 02:56:24+00:00


In [21]:
   print(df["stream_name"].value_counts())

stream_name
cisco_asa                              95062
aws_vpc_flow_log                       19219
microsoft-windows-security-auditing      129
Crowdstrike Detection                     58
barracuda_waf                             29
symantec_dlp                              14
generic_log                               10
Name: count, dtype: int64


In [19]:
   print(df["split"].value_counts())
   print(df.groupby([SRC, "split"]).size())

split
test       57921
train      28466
val        27651
dropped      483
Name: count, dtype: int64
stream_name                          split  
Crowdstrike Detection                dropped       58
aws_vpc_flow_log                     dropped      213
                                     test       13160
                                     train       3948
                                     val         1898
barracuda_waf                        dropped       29
cisco_asa                            dropped       30
                                     test       44761
                                     train      24518
                                     val        25753
generic_log                          dropped       10
microsoft-windows-security-auditing  dropped      129
symantec_dlp                         dropped       14
dtype: int64


In [20]:
cuts = {
    "cisco_asa": {
        "train": ("2024-07-26 11:00", "2024-07-26 14:00"),
        "val":   ("2024-07-26 17:00", "2024-07-26 21:00"),
        "test":  ("2024-07-27 00:00", "2024-07-27 07:00"),
    },
    "aws_vpc_flow_log": {
        "train": ("2024-07-26 00:00", "2024-07-28 00:00"),
        "val":   ("2024-07-29 16:00", "2024-07-29 19:00"),
        "test":  ("2024-07-29 20:00", "2024-07-31 00:00"),
    },
}

In [18]:
df["split"] = "dropped"
for src, sp in cuts.items():
    for name, (a, b) in sp.items():
        m = (df[SRC] == src) & (df["timestamp"] >= pd.Timestamp(a, tz="UTC")) & (df["timestamp"] < pd.Timestamp(b, tz="UTC"))
        df.loc[m, "split"] = name

print(df["split"].value_counts())
print(df.groupby([SRC, "split"]).size())


split
test       57921
train      28466
val        27651
dropped      483
Name: count, dtype: int64
stream_name                          split  
Crowdstrike Detection                dropped       58
aws_vpc_flow_log                     dropped      213
                                     test       13160
                                     train       3948
                                     val         1898
barracuda_waf                        dropped       29
cisco_asa                            dropped       30
                                     test       44761
                                     train      24518
                                     val        25753
generic_log                          dropped       10
microsoft-windows-security-auditing  dropped      129
symantec_dlp                         dropped       14
dtype: int64


In [25]:
print(df["label_binary"].value_counts())
print(df["incident_ids"].head(10))
print(df["incident_ids"].map(len).value_counts().head())

label_binary
suspicious    106793
malicious       7728
Name: count, dtype: int64
0    []
1    []
2    []
3    []
4    []
5    []
6    []
7    []
8    []
9    []
Name: incident_ids, dtype: str
incident_ids
2      106793
40       7128
80        572
120        28
Name: count, dtype: int64


In [26]:
print(df.loc[df.label_binary == "malicious", "incident_ids"].iloc[0])
print(type(df["incident_ids"].iloc[0]))

["e8a74c50-4d08-11ef-8afe-cb5955420555"]
<class 'str'>


In [27]:
import json
mal = df[df.label_binary == "malicious"].copy()
mal["ids"] = mal["incident_ids"].apply(json.loads)
mal = mal.explode("ids").rename(columns={"ids": "incident"})

In [28]:
m = mal[mal.split != "dropped"]
print(m.groupby([SRC, "split"]).agg(mal_rows=("timestamp", "size"),
                                    incidents=("incident", "nunique")))

                        mal_rows  incidents
stream_name      split                     
aws_vpc_flow_log test       3113         44
                 train       611          9
                 val         432         23
cisco_asa        test        973        640
                 train      2596        554
                 val         582        336


In [29]:
for src in ["cisco_asa", "aws_vpc_flow_log"]:
    s = m[m[SRC] == src]
    k = {sp: set(s.loc[s.split == sp, "incident"]) for sp in ["train", "val", "test"]}
    print(src, "train&val:", len(k["train"] & k["val"]),
          "| train&test:", len(k["train"] & k["test"]),
          "| val&test:", len(k["val"] & k["test"]))

cisco_asa train&val: 0 | train&test: 0 | val&test: 44
aws_vpc_flow_log train&val: 0 | train&test: 0 | val&test: 0


In [30]:
# incidents that appear in ASA val also appear in test -> drop from test
asa = m[m[SRC] == "cisco_asa"]
leak = set(asa.loc[asa.split == "val", "incident"]) & set(asa.loc[asa.split == "test", "incident"])
print(len(leak))

bad_rows = mal[(mal[SRC] == "cisco_asa") & (mal.split == "test") & (mal.incident.isin(leak))]
drop_idx = bad_rows.index.unique()          # row indices in the original df
df.loc[drop_idx, "split"] = "dropped"

print(df[df.label_binary == "malicious"].groupby([SRC, "split"]).size())

44
stream_name                          split  
Crowdstrike Detection                dropped      20
aws_vpc_flow_log                     test       3113
                                     train       611
                                     val         432
barracuda_waf                        dropped      11
cisco_asa                            dropped     163
                                     test        711
                                     train      2116
                                     val         535
microsoft-windows-security-auditing  dropped      16
dtype: int64


In [31]:
import json
mal = df[df.label_binary == "malicious"].copy()
mal["ids"] = mal["incident_ids"].apply(json.loads)
mal = mal.explode("ids").rename(columns={"ids": "incident"})
m = mal[mal.split != "dropped"]

for src in ["cisco_asa", "aws_vpc_flow_log"]:
    s = m[m[SRC] == src]
    k = {sp: set(s.loc[s.split == sp, "incident"]) for sp in ["train", "val", "test"]}
    print(src, "train&val:", len(k["train"] & k["val"]),
          "| train&test:", len(k["train"] & k["test"]),
          "| val&test:", len(k["val"] & k["test"]))

print(df[df.label_binary == "malicious"].groupby([SRC, "split"]).size())
print(m.groupby([SRC, "split"])["incident"].nunique())

cisco_asa train&val: 0 | train&test: 0 | val&test: 0
aws_vpc_flow_log train&val: 0 | train&test: 0 | val&test: 0
stream_name                          split  
Crowdstrike Detection                dropped      20
aws_vpc_flow_log                     test       3113
                                     train       611
                                     val         432
barracuda_waf                        dropped      11
cisco_asa                            dropped     163
                                     test        711
                                     train      2116
                                     val         535
microsoft-windows-security-auditing  dropped      16
dtype: int64
stream_name       split
aws_vpc_flow_log  test      44
                  train      9
                  val       23
cisco_asa         test     577
                  train    554
                  val      336
Name: incident, dtype: int64


In [32]:
df[df.split != "dropped"].to_parquet("../data/interim/pool_split.parquet")